# 7.5 GRPO（Group Relative Policy Optimization）

> 🕐 预估学习时间：45分钟

GRPO 是 DeepSeek-R1 提出的新一代对齐训练算法，通过组内相对奖励替代 PPO 中的价值网络（Critic），大幅降低训练复杂度和显存开销。

本节涵盖：
- GRPO 核心原理与 PPO 对比
- 组内相对优势估计
- 无 Critic 的强化学习
- 完整 GRPO 训练流程
- 与 DPO/PPO 的选择策略

## 1. GRPO vs PPO：为什么需要 GRPO？

**PPO 的问题**：
- 需要训练价值网络（Critic）来估计状态价值，额外占用显存
- Critic 的估计误差会传递给 Actor，影响训练稳定性
- 在大模型场景下，Critic 通常与 Actor 同等规模，训练成本翻倍

**GRPO 的解决方案**：
- 对每个 prompt 采样一组（Group）多个回复
- 用组内平均奖励作为基线（baseline），无需 Critic
- 组内归一化的优势函数替代 GAE
- 显存需求仅为 PPO 的约 50%

**核心公式**：

$$A_i = \frac{r_i - \text{mean}(\{r_1, ..., r_G\})}{\text{std}(\{r_1, ..., r_G\})}$$

$$L_{GRPO} = -\frac{1}{G}\sum_{i=1}^{G} \min\left(\rho_i A_i, \text{clip}(\rho_i, 1-\epsilon, 1+\epsilon) A_i\right) + \beta \cdot D_{KL}$$

其中 $\rho_i = \frac{\pi_\theta(y_i|x)}{\pi_{old}(y_i|x)}$ 为重要性采样比率

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class SimplePolicy(nn.Module):
    def __init__(self, d_model=128, vocab_size=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 256), nn.ReLU(),
            nn.LayerNorm(256),
            nn.Linear(256, vocab_size)
        )
    def forward(self, x):
        return self.net(x)

def compute_group_advantages(rewards, epsilon=1e-8):
    mean_r = rewards.mean(dim=0, keepdim=True)
    std_r = rewards.std(dim=0, keepdim=True)
    advantages = (rewards - mean_r) / (std_r + epsilon)
    return advantages

rewards = torch.tensor([
    [0.8, 0.3, 0.6],
    [0.5, 0.9, 0.7],
    [0.2, 0.4, 0.1],
    [0.6, 0.5, 0.8],
], dtype=torch.float)

advantages = compute_group_advantages(rewards)

print('=== GRPO Group Advantage ===')
print(f'Rewards (4 prompts x 3 responses):')
print(rewards)
print(f'\nGroup-relative advantages:')
print(advantages)
print(f'\nMean per group: {rewards.mean(dim=1).tolist()}')
print(f'Std per group:  {rewards.std(dim=1).tolist()}')
print(f'\nKey: Advantages are group-normalized - no value model needed!')
print(f'Good responses (above group mean) get positive advantages,')
print(f'bad responses (below group mean) get negative advantages.')

## 2. 奖励模型集成与规则奖励

DeepSeek-R1 使用两种奖励信号的组合：
- **结果奖励（Outcome Reward）**：规则化验证（如数学题答案正确性、代码执行结果）
- **过程奖励（Process Reward）**：奖励模型对推理步骤质量打分

**为什么规则奖励重要**：
- 数学/代码等任务有明确的正确性标准，无需奖励模型
- 规则奖励无奖励 hacking 风险
- 与奖励模型结合可兼顾正确性和格式

In [ ]:
class MixedReward:
    def __init__(self, rm_model=None, rule_weight=0.5):
        self.rm = rm_model
        self.rule_weight = rule_weight

    def rule_based_reward(self, response, ground_truth=None):
        rewards = torch.zeros(len(response))
        if ground_truth is not None:
            for i, (resp, gt) in enumerate(zip(response, ground_truth)):
                if torch.allclose(resp, gt, rtol=1e-2):
                    rewards[i] = 1.0
                else:
                    similarity = F.cosine_similarity(resp.unsqueeze(0), gt.unsqueeze(0))
                    rewards[i] = (similarity + 1) / 2
        return rewards

    def rm_reward(self, responses):
        if self.rm is None:
            return torch.zeros(len(responses))
        return self.rm(responses).detach()

    def compute(self, responses, ground_truth=None):
        rule_r = self.rule_based_reward(responses, ground_truth)
        rm_r = self.rm_reward(responses)
        return self.rule_weight * rule_r + (1 - self.rule_weight) * rm_r

responses = torch.randn(4, 128)
ground_truth = torch.randn(4, 128)
reward_fn = MixedReward(rule_weight=0.7)
total_rewards = reward_fn.compute(responses, ground_truth)

print('=== Mixed Reward (Rule + RM) ===')
print(f'Rule reward: {reward_fn.rule_based_reward(responses, ground_truth).tolist()}')
print(f'Combined reward: {total_rewards.tolist()}')
print(f'\nKey: Rule-based rewards for verifiable tasks eliminate reward hacking.')
print(f'DeepSeek-R1 uses rule rewards for math + RM for general quality.')

## 3. GRPO 完整训练流程

GRPO 的训练循环：
1. 采样 prompts batch
2. 对每个 prompt 用当前策略生成 G 个回复（group）
3. 计算每个回复的奖励（规则+RM）
4. 组内归一化得到优势函数
5. 计算 GRPO loss 并更新策略
6. 用 KL 惩罚约束不偏离参考模型

**与 PPO 的关键差异**：
| 组件 | PPO | GRPO |
|------|-----|------|
| 优势估计 | GAE + Value Network | 组内归一化 |
| Value Network | 需要 | 不需要 |
| 采样方式 | 单回复 | 组采样（G>1） |
| 显存需求 | ~2x 模型大小 | ~1.5x 模型大小 |

In [ ]:
class GRPOTrainer:
    def __init__(self, policy, ref_policy, reward_fn,
                 group_size=4, kl_coef=0.1, epsilon=0.2,
                 lr=1e-5, max_grad_norm=1.0):
        self.policy = policy
        self.ref_policy = ref_policy
        self.reward_fn = reward_fn
        self.group_size = group_size
        self.kl_coef = kl_coef
        self.epsilon = epsilon
        self.max_grad_norm = max_grad_norm
        self.optimizer = torch.optim.AdamW(policy.parameters(), lr=lr)

        for p in self.ref_policy.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def sample_group(self, prompts):
        B, D = prompts.shape
        vocab_size = self.policy.net[-1].out_features
        logits = self.policy(prompts)
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()

        responses = []
        old_log_probs_list = []
        for _ in range(self.group_size):
            actions = probs.multinomial(1).squeeze(-1)
            action_log_probs = log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
            responses.append(F.one_hot(actions, vocab_size).float().sum(dim=1))
            old_log_probs_list.append(action_log_probs)

        responses = torch.stack(responses, dim=0)
        old_log_probs = torch.stack(old_log_probs_list, dim=0)
        return responses, old_log_probs

    @torch.no_grad()
    def compute_ref_log_probs(self, prompts, actions):
        ref_logits = self.ref_policy(prompts)
        ref_log_probs = F.log_softmax(ref_logits, dim=-1)
        return ref_log_probs

    def train_step(self, prompts):
        B = prompts.shape[0]
        G = self.group_size

        responses, old_log_probs = self.sample_group(prompts)
        rewards = torch.zeros(G, B)
        for g in range(G):
            rewards[g] = self.reward_fn.compute(responses[g])

        advantages = compute_group_advantages(rewards)

        total_loss = 0
        total_kl = 0
        n_inner_epochs = 2

        for _ in range(n_inner_epochs):
            logits = self.policy(prompts)
            log_probs = F.log_softmax(logits, dim=-1)

            policy_loss = 0
            kl_sum = 0

            for g in range(G):
                actions = responses[g].argmax(dim=-1)
                new_log_probs = log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
                ref_log_probs = self.compute_ref_log_probs(prompts, actions)

                ratio = (new_log_probs - old_log_probs[g]).exp()

                surr1 = ratio * advantages[g]
                surr2 = torch.clamp(ratio, 1 - self.epsilon, 1 + self.epsilon) * advantages[g]
                policy_loss += -torch.min(surr1, surr2).mean()

                kl_sum += (ref_log_probs - new_log_probs).mean()

            kl_div = kl_sum / G
            loss = policy_loss / G + self.kl_coef * kl_div

            self.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
            self.optimizer.step()

            total_loss += loss.item()
            total_kl += kl_div.item()

        return {
            'loss': total_loss / n_inner_epochs,
            'kl': total_kl / n_inner_epochs,
            'mean_reward': rewards.mean().item(),
            'reward_std': rewards.std().item(),
        }

policy = SimplePolicy(d_model=128, vocab_size=100)
ref_policy = SimplePolicy(d_model=128, vocab_size=100)
ref_policy.load_state_dict(policy.state_dict())
reward_fn = MixedReward(rule_weight=0.7)

trainer = GRPOTrainer(policy, ref_policy, reward_fn, group_size=4)

prompts = torch.randn(16, 128)
gt = torch.randn(16, 128)

print('=== GRPO Training Loop ===')
for step in range(25):
    metrics = trainer.train_step(prompts)
    if (step + 1) % 5 == 0:
        print(f'Step {step+1:3d}: loss={metrics["loss"]:.4f}, '
              f'kl={metrics["kl"]:.4f}, '
              f'reward={metrics["mean_reward"]:.3f}±{metrics["reward_std"]:.3f}')

print(f'\nKey: GRPO uses group-relative advantages, NO value network needed.')
print(f'This is the key innovation of DeepSeek-R1 over traditional RLHF/PPO.')
print(f'Group sampling provides a natural baseline - the average of peers.')

## 4. GRPO 在 DeepSeek-R1 中的实践

**DeepSeek-R1 的两阶段训练**：

**阶段 1：R1-Zero（纯 RL）**
- 从 DeepSeek-V3-Base 出发
- 仅用 GRPO + 规则奖励（数学正确性、格式合规）
- 涌现出：自我反思、多步验证、自动纠错等推理能力
- 问题：输出可读性差、语言混杂

**阶段 2：R1（冷启动 + RL）**
- 先用少量高质量思维链数据做冷启动 SFT
- 再用 GRPO 进行 RL 训练，加入语言一致性奖励
- 最终模型兼具推理能力和可读性

**关键发现**：
- 纯 RL 足以催生推理能力（无需精心设计的 prompt）
- 组采样大小 G 越大，优势估计越稳定
- 规则奖励避免了奖励 hacking

In [ ]:
def compare_grpo_ppo_memory(d_model=4096, n_layers=32, vocab_size=32000):
    model_params = d_model * d_model * n_layers * 4
    model_gb = model_params * 4 / 1e9

    ppo_critic_gb = model_gb
    ppo_total_gb = model_gb * 2 + model_gb * 2 + model_gb * 2
    grpo_total_gb = model_gb * 2 + model_gb * 2

    print('=== Memory Comparison: PPO vs GRPO ===')
    print(f'Model size: ~{model_gb:.1f} GB (FP32)')
    print(f'\nPPO components:')
    print(f'  Actor model:     ~{model_gb:.1f} GB')
    print(f'  Reference model: ~{model_gb:.1f} GB')
    print(f'  Critic model:    ~{ppo_critic_gb:.1f} GB')
    print(f'  Optimizer:       ~{model_gb*2:.1f} GB')
    print(f'  PPO Total:       ~{ppo_total_gb:.1f} GB')
    print(f'\nGRPO components:')
    print(f'  Actor model:     ~{model_gb:.1f} GB')
    print(f'  Reference model: ~{model_gb:.1f} GB')
    print(f'  Optimizer:       ~{model_gb*2:.1f} GB')
    print(f'  GRPO Total:      ~{grpo_total_gb:.1f} GB')
    print(f'\n  Memory savings: ~{(1-grpo_total_gb/ppo_total_gb)*100:.0f}%')
    print(f'  (No critic, no critic optimizer)')

compare_grpo_ppo_memory()

print(f'\nKey: GRPO eliminates the critic model entirely.')
print(f'For large models, this is a massive memory saving.')

## 5. 对齐方法选型指南

| 方法 | 计算成本 | 需要 RM | 需要 Critic | 在线采样 | 适用场景 |
|------|----------|---------|-------------|----------|----------|
| DPO | 低 | 否 | 否 | 否 | 有充足偏好数据 |
| PPO/RLHF | 高 | 是 | 是 | 是 | 追求最优、有资源 |
| **GRPO** | **中** | **可选** | **否** | **是** | **推理任务对齐** |
| ORPO | 低 | 否 | 否 | 否 | SFT+对齐一步完成 |
| SimPO | 低 | 否 | 否 | 否 | 无参考模型场景 |

**选择 GRPO 当**：
- 任务有可验证的奖励信号（数学、代码）
- 希望节省 Critic 的显存开销
- 需要在线探索来发现更好的策略

**选择 DPO 当**：
- 有大量高质量偏好数据
- 计算资源有限
- 不需要在线探索

## 📝 课后思考题

1. GRPO 的组采样大小 G 如何影响优势估计的方差？G 越大越好吗？
2. 如果奖励信号噪声很大，GRPO 的组内归一化会有什么问题？
3. DeepSeek-R1 为什么先做冷启动 SFT 再做 GRPO，而不是直接 GRPO？
4. 如何将 GRPO 应用到开放式对话任务（没有明确对错的任务）？